# Swin student results

Compare extraction quality, parameter count, and encoder latency for the teacher and students.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESEARCH_ROOT = Path("/domino/datasets/local/donut/research")
EVALUATION_DIR = RESEARCH_ROOT / "results/evaluation"
LATENCY_DIR = RESEARCH_ROOT / "results/encoder_latency"

## Quality and parameters

In [ ]:
records = {}
for path in sorted(EVALUATION_DIR.glob("*.json")):
    record = json.loads(path.read_text())
    name = record["config"]["name"]
    summary = record["summary"]
    records[name] = {
        "strict_macro_field_f1": summary["strict_macro_field_f1"],
        "total_parameters": summary["parameters"]["total"],
        "encoder_parameters": summary["parameters"]["encoder"],
        "decoder_parameters": summary["parameters"]["decoder"],
    }

comparison = pd.DataFrame.from_dict(records, orient="index")
teacher = comparison.loc["teacher"]
comparison["parameter_reduction"] = (
    1 - comparison["total_parameters"] / teacher["total_parameters"]
)
comparison["score_change"] = (
    comparison["strict_macro_field_f1"] - teacher["strict_macro_field_f1"]
)
comparison.sort_values("total_parameters", ascending=False)

In [ ]:
ordered = comparison.sort_values("total_parameters")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ordered["strict_macro_field_f1"].plot.barh(ax=axes[0])
axes[0].set(title="Extraction quality", xlabel="Strict macro field F1", xlim=(0, 1))
(ordered["total_parameters"] / 1e6).plot.barh(ax=axes[1])
axes[1].set(title="Model size", xlabel="Millions of parameters")
plt.tight_layout()

## Encoder latency

In [ ]:
latency_records = []
for path in sorted(LATENCY_DIR.glob("*.json")):
    record = json.loads(path.read_text())
    config = record["config"]
    for result in record["results"]:
        latency_records.append(
            {
                "name": config["name"],
                "image_height": config["image_height"],
                "image_width": config["image_width"],
                "batch_size": result["batch_size"],
                "encoder_latency_ms": result["encoder_latency_ms"],
            }
        )

latency = pd.DataFrame(latency_records)
teacher_latency = latency[latency["name"] == "teacher"].rename(
    columns={"encoder_latency_ms": "teacher_latency_ms"}
)
latency = latency.merge(
    teacher_latency[
        ["image_height", "image_width", "batch_size", "teacher_latency_ms"]
    ],
    on=["image_height", "image_width", "batch_size"],
)
latency["speedup_vs_teacher"] = (
    latency["teacher_latency_ms"] / latency["encoder_latency_ms"]
)
latency.sort_values(["image_height", "batch_size", "encoder_latency_ms"])

In [ ]:
image_sizes = (
    latency[["image_height", "image_width"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
image_sizes = list(image_sizes)
fig, axes = plt.subplots(
    1, len(image_sizes), figsize=(6 * len(image_sizes), 4), squeeze=False
)

for axis, (height, width) in zip(axes[0], image_sizes):
    rows = latency[
        (latency["image_height"] == height) & (latency["image_width"] == width)
    ]
    for name, model_rows in rows.groupby("name"):
        model_rows = model_rows.sort_values("batch_size")
        axis.plot(
            model_rows["batch_size"],
            model_rows["encoder_latency_ms"],
            marker="o",
            label=name,
        )
    axis.set(
        title=f"{height}×{width}", xlabel="Batch size", ylabel="Encoder latency (ms)"
    )
    axis.grid(alpha=0.3)

axes[0, -1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

## Quality, size, and latency trade-off

Choose one resolution and batch size for a direct model comparison.

In [ ]:
SELECTED_IMAGE_SIZE = (1280, 960)
SELECTED_BATCH_SIZE = 1

height, width = SELECTED_IMAGE_SIZE
selected_latency = latency[
    (latency["image_height"] == height)
    & (latency["image_width"] == width)
    & (latency["batch_size"] == SELECTED_BATCH_SIZE)
].set_index("name")
tradeoff = comparison.join(
    selected_latency[["encoder_latency_ms", "speedup_vs_teacher"]]
)
tradeoff.sort_values("encoder_latency_ms")

In [ ]:
fig, axis = plt.subplots(figsize=(8, 5))
axis.scatter(tradeoff["encoder_latency_ms"], tradeoff["strict_macro_field_f1"])
for name, row in tradeoff.iterrows():
    axis.annotate(
        str(name), (row["encoder_latency_ms"], row["strict_macro_field_f1"]), fontsize=8
    )
axis.set(
    xlabel="Encoder latency (ms)",
    ylabel="Strict macro field F1",
    title=f"Quality/latency trade-off: {height}×{width}, batch {SELECTED_BATCH_SIZE}",
)
axis.grid(alpha=0.3)
plt.tight_layout()